In [3]:
from pyspark.sql import SparkSession

In [6]:
from pyspark.sql import functions as sf

In [4]:
spark=SparkSession.builder.appName("set123").getOrCreate()

##SET1

SELECT name , age from employees where age> 25;

In [5]:
#small sample dataframe
df=spark.createDataFrame(
    [
        ("Ram" , 25),
        ("Sita" , 30),
        ("Ravan" , 15)
    ] , ["name" , "age"]
)

In [9]:
result=df.filter(sf.col("age")>25).select("name" , "age")

In [10]:
result.show()

+----+---+
|name|age|
+----+---+
|Sita| 30|
+----+---+



SELECT department, COUNT(*) AS employee_count
FROM employees
WHERE salary > 50000
GROUP BY department;


In [12]:
#small sample dataframe
df=spark.createDataFrame(
    [
        (101 , "IT" , 45000),
        (102 , "HR" , 50000),
        (103 , "IT" , 60000),
        (104 , "HR" , 55000),
        (105 , "IT" , 55000)
    ] , ["emp_id" , "department" , "salary"]
)

In [13]:
df.filter(sf.col("salary")>50000)\
.groupBy("department")\
.count()\
.select("department" , sf.col("count").alias("employee_count"))\
.show()

+----------+--------------+
|department|employee_count|
+----------+--------------+
|        HR|             1|
|        IT|             2|
+----------+--------------+



SELECT department,
       AVG(salary) AS avg_salary,
       MAX(salary) AS max_salary
FROM employees
WHERE age >= 25
GROUP BY department
HAVING AVG(salary) > 60000
ORDER BY avg_salary DESC;


In [21]:
#small sample dataframe
df=spark.createDataFrame(
    [
        (101 , "IT" , 45000 , 24),
        (102 , "HR" , 50000 , 26),
        (103 , "IT" , 690000 , 30),
        (104 , "HR" , 20000 , 35),
        (105 , "IT" , 155000 , 40)
    ] , ["emp_id" , "department" , "salary" , "age"]
)

In [22]:
df.filter(sf.col("age")>=25).groupBy("department").agg(sf.avg(sf.col("salary")).alias("avg_salary") , sf.max(sf.col("salary")).alias("max_salary")).filter(sf.col("avg_salary")>60000).orderBy(sf.col("avg_salary").desc()).select("department" , "avg_salary" , "max_salary").show()

+----------+----------+----------+
|department|avg_salary|max_salary|
+----------+----------+----------+
|        IT|  422500.0|    690000|
+----------+----------+----------+



##SET 2


SELECT product_name, price
FROM products
WHERE price BETWEEN 100 AND 500
ORDER BY price DESC;


In [26]:
#small dataframe
df=spark.createDataFrame([
    ("nothing phone" , 500),
    ("laptop" , 350),
    ("samsung phone" , 10)
] ,
[
    "product_name" , "price"
])

In [27]:
df.filter(sf.col("price").between(100 , 500)).select("product_name" , "price").orderBy(sf.col("price").desc()).show()

+-------------+-----+
| product_name|price|
+-------------+-----+
|nothing phone|  500|
|       laptop|  350|
+-------------+-----+



SELECT department,
       COUNT(*) AS employee_count,
       AVG(salary) AS average_salary
FROM employees
GROUP BY department
ORDER BY average_salary DESC;


In [30]:
#small sample dataframe
df=spark.createDataFrame(
    [
        (101 , "IT" , 45000 , 24),
        (102 , "HR" , 50000 , 26),
        (103 , "IT" , 690000 , 30),
        (104 , "HR" , 20000 , 35),
        (105 , "IT" , 155000 , 40)
    ] , ["emp_id" , "department" , "salary" , "age"]
)

In [31]:
df.groupBy("department").agg(sf.count(sf.col("emp_id")).alias("employee_count") , sf.avg(sf.col("salary")).alias("average_salary")).select("department" , "employee_count" , "average_salary").orderBy(sf.col("average_salary").desc()).show()

+----------+--------------+-----------------+
|department|employee_count|   average_salary|
+----------+--------------+-----------------+
|        IT|             3|296666.6666666667|
|        HR|             2|          35000.0|
+----------+--------------+-----------------+



SELECT e.name,
       e.department,
       d.location,
       e.salary
FROM employees e
JOIN departments d
    ON e.department_id = d.department_id
WHERE e.salary > 60000
  AND d.location = 'Bangalore'
ORDER BY e.salary DESC;


In [33]:
#small sample dataframe
e_df=spark.createDataFrame(
    [
        ("ram" , 101 , "IT" , 45000 ),
        ("sita" , 102 , "HR" , 50000 ),
        ("lakshman" , 101 ,  "IT" , 690000 ),
        ("ravan" , 102 ,  "HR" , 20000 ),
        ("baratha" , 101 , "IT" , 155000 )
    ] , ["name" , "department_id" , "department" , "salary"]
)
d_df=spark.createDataFrame(
    [
        (101 , "Bangalore"),
        (102, "BANGALORE")
    ] , ["department_id" , "location"]
)

In [37]:
e_df.join(d_df , on="department_id" , how="inner").filter((sf.col("salary")>60000) & (sf.trim(sf.initcap(sf.col("location")))=="Bangalore")).select("name" , "department" , "location" , "salary").orderBy(sf.col("salary").desc()).show()

+--------+----------+---------+------+
|    name|department| location|salary|
+--------+----------+---------+------+
|lakshman|        IT|Bangalore|690000|
| baratha|        IT|Bangalore|155000|
+--------+----------+---------+------+



## SET3

SELECT name,
       salary,
       salary * 1.10 AS revised_salary
FROM employees;


In [39]:
#small sample dataframe
df=spark.createDataFrame(
    [
        ("ram" ,  45000),
        ("sita" ,  50000),
        ("lakshman" ,  690000),
        ("ravan" ,  20000),
        ("baratha" ,  155000)
    ] , ["name" , "salary"]
)

In [42]:
df.select("name" , "salary" , (sf.col("salary")*1.10).alias("revised_salary")).show()

+--------+------+-----------------+
|    name|salary|   revised_salary|
+--------+------+-----------------+
|     ram| 45000|49500.00000000001|
|    sita| 50000|55000.00000000001|
|lakshman|690000|759000.0000000001|
|   ravan| 20000|          22000.0|
| baratha|155000|         170500.0|
+--------+------+-----------------+



SELECT department,
       COUNT(*) AS employee_count
FROM employees
WHERE salary IS NOT NULL
GROUP BY department
HAVING COUNT(*) >= 5
ORDER BY employee_count DESC;


In [47]:
#small sample dataframe
df=spark.createDataFrame(
    [
        (101 , "IT" , 45000 , 24),
        (102 , "HR" , 50000 , 26),
        (103 , "IT" , 690000 , 30),
        (104 , "HR" , 20000 , 35),
        (105 , "IT" , 155000 , 40),
        (106 , "IT" , 45000 , 24),
        (107 , "HR" , 50000 , 26),
        (108 , "IT" , 690000 , 30),
        (109 , "HR" , 20000 , 35),
        (110 , "IT" , 155000 , 40)
    ] , ["emp_id" , "department" , "salary" , "age"]
)

In [48]:
df.filter(sf.col("salary").isNotNull()).groupBy("department").agg(sf.count(sf.col("emp_id")).alias("employee_count")).filter(sf.col("employee_count")>=5).orderBy(sf.col("employee_count").desc()).select("department" , "employee_count").show()

+----------+--------------+
|department|employee_count|
+----------+--------------+
|        IT|             6|
+----------+--------------+



SELECT
    e.department,
    COUNT(DISTINCT e.employee_id) AS employee_count,
    AVG(e.salary) AS avg_salary,
    MAX(e.salary) AS highest_salary
FROM employees e
JOIN departments d
    ON e.department_id = d.department_id
WHERE e.salary IS NOT NULL
  AND d.location IN ('Bangalore', 'Hyderabad')
GROUP BY e.department
HAVING AVG(e.salary) > 60000
ORDER BY avg_salary DESC;


In [49]:
#small sample dataframe
e_df=spark.createDataFrame(
    [
        ("ram" , 101 , "IT" , 45000 ),
        ("sita" , 102 , "HR" , 50000 ),
        ("lakshman" , 101 ,  "IT" , 690000 ),
        ("ravan" , 102 ,  "HR" , 20000 ),
        ("baratha" , 101 , "IT" , 155000 )
    ] , ["name" , "department_id" , "department" , "salary"]
)
d_df=spark.createDataFrame(
    [
        (101 , "Bangalore"),
        (102, "BANGALORE")
    ] , ["department_id" , "location"]
)

In [53]:
e_df.join(d_df , on="department_id" , how="inner").filter((sf.col("salary").isNotNull()) & (sf.initcap(sf.col("location"))).isin('Bangalore' , 'Hyderabad'))\
.groupBy("department").agg(sf.max(sf.col("salary")).alias("highest_salary") , sf.avg(sf.col("salary")).alias("avg_salary")
,sf.countDistinct(sf.col("name")).alias("employee_count")
).filter(sf.col("avg_salary")>60000).orderBy(sf.col("avg_salary").desc()).show()

+----------+--------------+-----------------+--------------+
|department|highest_salary|       avg_salary|employee_count|
+----------+--------------+-----------------+--------------+
|        IT|        690000|296666.6666666667|             3|
+----------+--------------+-----------------+--------------+

